# Day 37 — **tenacity: Retry & Circuit Breaker**

**Phase 2 · Module 5: Multi-Agent Orchestration** · ~30 min

The module notebook built retry/backoff/breaker by hand to show the mechanics. In production you don't hand-roll them — you use **`tenacity`**, the standard Python retry library. It turns a page of loop-and-sleep code into one decorator, and it gets the subtle parts right (jitter, which exceptions to retry, what to do when retries run out).

Real `tenacity` here. Install: `pip install tenacity`.

Today:
1. `@retry` with **exponential backoff + jitter** — the correct default.
2. **Retry only transient errors** — `retry_if_exception_type`.
3. **Give up cleanly** — `retry_error_callback` + `stop_after_attempt`.
4. A **circuit breaker** wrapping tenacity — stop calling a dead dependency.

## 1. `@retry` with exponential backoff + jitter

One decorator replaces the whole retry loop. `wait_exponential` spaces attempts out; `wait_random` adds jitter so a fleet of agents doesn't retry in lockstep.

In [1]:
from tenacity import (retry, stop_after_attempt, wait_exponential, wait_random,
                      retry_if_exception_type, before_sleep_log)
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("agent")

class ThrottleError(Exception):
    """Transient — Bedrock 429 / service 503."""

_calls = {"n": 0}
def flaky_bedrock():
    _calls["n"] += 1
    if _calls["n"] < 3:
        raise ThrottleError(f"429 throttled (attempt {_calls['n']})")
    return {"grade": "BBB"}

@retry(stop=stop_after_attempt(5),
       wait=wait_exponential(multiplier=0.05, max=1) + wait_random(0, 0.05),
       before_sleep=before_sleep_log(log, logging.INFO))
def call_bedrock():
    return flaky_bedrock()

print("result:", call_bedrock(), "| attempts:", _calls["n"])

Retrying __main__.call_bedrock in 0.0966 seconds as it raised ThrottleError: 429 throttled (attempt 1).


Retrying __main__.call_bedrock in 0.127 seconds as it raised ThrottleError: 429 throttled (attempt 2).


result: {'grade': 'BBB'} | attempts: 3


☝ `wait_exponential(...) + wait_random(...)` composes two wait strategies — exponential base with additive jitter — which is the "full jitter" idea from the module notebook, expressed declaratively. `before_sleep_log` gives you free observability: every retry is logged with the attempt number and delay, so a retry storm shows up in your logs instead of silently inflating latency. This one decorator is what you actually put on every external call.

## 2. Retry only transient errors

The cardinal rule: retry a 429/timeout, **never** a 400/validation error — retrying a deterministic failure just wastes attempts and money. `retry_if_exception_type` whitelists what's worth retrying.

In [2]:
class ValidationError(Exception):
    """Permanent — malformed request. Retrying will NEVER help."""

def call_with_bad_input(_c={"n": 0}):
    _c["n"] += 1
    raise ValidationError("field 'amount' must be > 0")

@retry(retry=retry_if_exception_type(ThrottleError),   # ONLY ThrottleError is retried
       stop=stop_after_attempt(5), wait=wait_exponential(multiplier=0.01))
def guarded():
    return call_with_bad_input()

import time
t0 = time.perf_counter()
try:
    guarded()
except ValidationError as e:
    print(f"failed immediately (no retries): {e}  [{(time.perf_counter()-t0)*1000:.0f}ms]")

failed immediately (no retries): field 'amount' must be > 0  s]


☝ Because `ValidationError` isn't in the `retry_if_exception_type` whitelist, tenacity re-raises it on the **first** attempt — no backoff, no five wasted round-trips. This is the single most common retry bug in production: a blanket `except Exception: retry` that hammers a service with a request it will always reject. Whitelist transient exceptions explicitly; let permanent ones fail fast.

## 3. Give up cleanly

When retries are exhausted you have two choices: re-raise (default), or return a **fallback** via `retry_error_callback`. For an agent, a graceful degraded answer often beats an exception (Day 37 module notebook).

In [3]:
def always_down(_c={"n": 0}):
    _c["n"] += 1
    raise ThrottleError("service still down")

def use_cached_fallback(retry_state):
    """Called when all attempts fail — return a degraded result instead of raising."""
    log.info(f"  all {retry_state.attempt_number} attempts failed -> cached fallback")
    return {"grade": "UNKNOWN", "source": "cache", "degraded": True}

@retry(retry=retry_if_exception_type(ThrottleError),
       stop=stop_after_attempt(3), wait=wait_exponential(multiplier=0.01),
       retry_error_callback=use_cached_fallback)
def score_with_fallback():
    return always_down()

print("degraded result:", score_with_fallback())

  all 3 attempts failed -> cached fallback


degraded result: {'grade': 'UNKNOWN', 'source': 'cache', 'degraded': True}


☝ `retry_error_callback` converts "exhausted all retries" from an exception into a **value** — here a cached, explicitly-`degraded` result. That's the Day-37 graceful-degradation principle wired into the retry decorator itself: the agent gets a usable answer flagged as low-confidence, rather than a stack trace. The `degraded: True` marker is essential — downstream (and any human gate) must be able to tell a real score from a fallback.

## 4. Circuit breaker on top of tenacity

Retry handles a blip; if the dependency is *down*, retrying every call wastes time. A breaker trips after N failures and fails fast for a cooldown. tenacity retries within a call; the breaker decides whether to call at all.

In [4]:
class CircuitOpen(Exception): pass

class Breaker:
    def __init__(self, threshold=3, cooldown=1.0):
        self.threshold, self.cooldown = threshold, cooldown
        self.fails, self.open_until = 0, 0.0
    def __call__(self, fn):
        def wrapper(*a, **k):
            if time.time() < self.open_until:
                raise CircuitOpen("circuit OPEN — failing fast")
            try:
                out = fn(*a, **k)
            except Exception:
                self.fails += 1
                if self.fails >= self.threshold:
                    self.open_until = time.time() + self.cooldown
                raise
            self.fails = 0
            return out
        return wrapper

breaker = Breaker(threshold=3, cooldown=5.0)

@breaker                                                # breaker outside...
@retry(retry=retry_if_exception_type(ThrottleError),   # ...retry inside
       stop=stop_after_attempt(2), wait=wait_exponential(multiplier=0.001),
       reraise=True)
def protected_call(_c={"n": 0}):
    _c["n"] += 1
    raise ThrottleError("down")

for i in range(5):
    try:
        protected_call()
    except (ThrottleError, CircuitOpen) as e:
        print(f"call {i}: {type(e).__name__}: {e}")

call 0: ThrottleError: down
call 1: ThrottleError: down
call 2: ThrottleError: down
call 3: CircuitOpen: circuit OPEN — failing fast
call 4: CircuitOpen: circuit OPEN — failing fast


☝ Order matters: **breaker outside, retry inside**. Each `protected_call` retries twice (tenacity, `reraise=True` so the real error surfaces); each *exhausted* call counts as one failure to the breaker. After 3 such failures the breaker opens and subsequent calls raise `CircuitOpen` **instantly** — no retries, no waiting on a dead service. Retry absorbs transient blips; the breaker stops you from retrying into an outage. Together they're the standard resilience stack for every agent tool call.

## Recap

| Goal | tenacity |
|---|---|
| Backoff + jitter | `wait_exponential(...) + wait_random(...)` |
| Cap attempts | `stop_after_attempt(n)` |
| Retry only transient | `retry_if_exception_type(...)` |
| Observe retries | `before_sleep_log` |
| Degrade on exhaustion | `retry_error_callback` |
| Stop calling a dead dep | circuit breaker *outside* retry |

**Rule:** `@retry` for blips, breaker for outages, whitelist transient exceptions, always degrade or re-raise deliberately. Tomorrow (Day 38): strict Pydantic output contracts so agents can't hand each other malformed data.